In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
SupportsFloat = float
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- metrics_bootstrap_filter ---
FIX_METRICS_BOOTSTRAP_FILTER_CONDITION_A = "plaintext"
FIX_METRICS_BOOTSTRAP_FILTER_CONDITION_B = "mixed"
FIX_METRICS_BOOTSTRAP_FILTER_METRIC = "mse"

# --- metrics_groupby_agg ---
FIX_METRICS_GROUPBY_AGG_METRIC = "mse"

# --- metrics_groupby_partition ---

print("✅ Fixtures loaded")
DF_METRICS_PD = pd.DataFrame({
    "condition": ["plaintext", "mixed", "plaintext", "mixed", "plaintext"],
    "score": [0.5, 0.7, 0.4, 0.8, 0.6],
    "mse": [0.05, 0.03, 0.06, 0.02, 0.04],
    "mae": [0.20, 0.15, 0.25, 0.10, 0.18],
})
DF_METRICS_PL = pl.from_pandas(DF_METRICS_PD)
df = DF_METRICS_PD


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_metrics_bootstrap_filter(condition_a, condition_b, metric):
    a = df[df["condition"] == condition_a][metric].to_numpy()
    b = df[df["condition"] == condition_b][metric].to_numpy()
    return b

def before_metrics_groupby_agg(metric):
    summary = (
        df.groupby("condition")[metric]
        .agg(["mean", "median"])
        .rename(columns={"mean": "mean", "median": "median"})
        .astype(float)
    )
    mixed_mean_raw = cast(SupportsFloat, summary.loc["mixed", "mean"])
    plaintext_mean_raw = cast(SupportsFloat, summary.loc["plaintext", "mean"])
    mixed_mean = float(mixed_mean_raw)
    plaintext_mean = float(plaintext_mean_raw)
    return plaintext_mean

def before_metrics_groupby_partition():
    def summarize_metrics(df: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
        """Compute descriptive statistics for each metric by condition."""
        stats = []
        for cond, group in df.groupby("condition"):
            for metric in metrics:
                series = group[metric]
                stats.append(
                    {
                        "condition": cond,
                        "metric": metric,
                        "mean": series.mean(),
                        "median": series.median(),
                        "std": series.std(),
                        "q10": series.quantile(0.1),
                        "q90": series.quantile(0.9),
                    }
                )
        return pd.DataFrame(stats)
    return summarize_metrics

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_metrics_bootstrap_filter(condition_a, condition_b, metric):

    a = df.filter(pl.col("condition") == condition_a).get_column(metric).to_numpy()
    b = df.filter(pl.col("condition") == condition_b).get_column(metric).to_numpy()
    return b

def gen_metrics_groupby_agg(metric):
    from typing import cast, SupportsFloat

    summary = (
        df.group_by("condition")
        .agg(
            pl.col(metric).mean().alias("mean"),
            pl.col(metric).median().alias("median"),
        )
        .rename({"mean": "mean", "median": "median"})
        .with_columns(pl.all().cast(float))
    )

    mixed_mean_raw = cast(
        SupportsFloat, summary.filter(pl.col("condition") == "mixed").select("mean").item()
    )
    plaintext_mean_raw = cast(
        SupportsFloat,
        summary.filter(pl.col("condition") == "plaintext").select("mean").item(),
    )
    mixed_mean = float(mixed_mean_raw)
    plaintext_mean = float(plaintext_mean_raw)
    return plaintext_mean

def gen_metrics_groupby_partition():


    def summarize_metrics(df: pl.DataFrame, metrics: list[str]) -> pl.DataFrame:
        """Compute descriptive statistics for each metric by condition."""
        stats = []
        for cond, group in df.filter(pl.col("condition").is_not_null()).sort("condition").group_by("condition", maintain_order=True):
            if isinstance(cond, tuple):
                cond = cond[0]
            for metric in metrics:
                series = group[metric]
                stats.append(
                    {
                        "condition": cond,
                        "metric": metric,
                        "mean": series.mean(),
                        "median": series.median(),
                        "std": series.std(),
                        "q10": series.quantile(0.1, interpolation="linear"),
                        "q90": series.quantile(0.9, interpolation="linear"),
                    }
                )
        return pl.DataFrame(stats)
    return summarize_metrics

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: metrics_bootstrap_filter ===

try:
    _old_df = df
    df = DF_METRICS_PL
    _r = gen_metrics_bootstrap_filter(FIX_METRICS_BOOTSTRAP_FILTER_CONDITION_A, FIX_METRICS_BOOTSTRAP_FILTER_CONDITION_B, FIX_METRICS_BOOTSTRAP_FILTER_METRIC)
    print("✅ L1 smoke gen_metrics_bootstrap_filter: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_metrics_bootstrap_filter: {type(_e).__name__}: {_e}")
finally:
    df = _old_df

try:
    _old_df = df
    df = DF_METRICS_PD
    _rb = before_metrics_bootstrap_filter(FIX_METRICS_BOOTSTRAP_FILTER_CONDITION_A, FIX_METRICS_BOOTSTRAP_FILTER_CONDITION_B, FIX_METRICS_BOOTSTRAP_FILTER_METRIC)
    print("✅ L1 smoke before_metrics_bootstrap_filter: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_metrics_bootstrap_filter: {type(_e).__name__}: {_e}")
finally:
    df = _old_df

try:
    _old_df = df
    df = DF_METRICS_PD
    _rb = before_metrics_bootstrap_filter(FIX_METRICS_BOOTSTRAP_FILTER_CONDITION_A, FIX_METRICS_BOOTSTRAP_FILTER_CONDITION_B, FIX_METRICS_BOOTSTRAP_FILTER_METRIC)
    df = DF_METRICS_PL
    _rg = gen_metrics_bootstrap_filter(FIX_METRICS_BOOTSTRAP_FILTER_CONDITION_A, FIX_METRICS_BOOTSTRAP_FILTER_CONDITION_B, FIX_METRICS_BOOTSTRAP_FILTER_METRIC)
    if np.allclose(_rb, _rg):
        print("✅ L2 equivalence metrics_bootstrap_filter: MATCH")
    else:
        print(f"❌ L2 equivalence metrics_bootstrap_filter: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L2 equivalence metrics_bootstrap_filter: setup error — {type(_e).__name__}: {_e}")
finally:
    df = _old_df

# L3 branch – alternate metric path
try:
    _old_df = df
    df = DF_METRICS_PD
    _before_edge = before_metrics_bootstrap_filter("plaintext", "mixed", "mae")
    df = DF_METRICS_PL
    _gen_edge = gen_metrics_bootstrap_filter("plaintext", "mixed", "mae")
    if np.allclose(_before_edge, _gen_edge):
        print("✅ L3 branch metrics_bootstrap_filter: MATCH")
    else:
        print(f"❌ L3 branch metrics_bootstrap_filter: MISMATCH — before={_before_edge}, gen={_gen_edge}")
except Exception as _e:
    print(f"❌ L3 branch metrics_bootstrap_filter: {type(_e).__name__}: {_e}")
finally:
    df = _old_df
